# 2. Exponentiating the Hamiltonian: the WII and VD2 MPO constructions

Here we build $U(\delta t)=e^{-iH\delta t}$ directly as a uniform (translation-invariant) MPO from a finite-state-machine (FSM) encoding of $H$, benchmark it against TDVP, and identify where it starts to break down.

The construction is provided by `expmpo` from the `ITensorExpMPO` package
(`ITensorExpMPOv2.jl/`, a fork of [tipfom/ITensorExpMPO.jl](https://github.com/tipfom/ITensorExpMPO.jl);
all of it is upstream work except the **VD2** second-order kernel, added for this thesis). In the
library it is wrapped as `expH_alcaraz(sites, λ, p; dt, mpo_alg)` with `mpo_alg ∈ {"WI","WII","VD2"}`.

## The Hamiltonian as a finite-state machine

A short-range Hamiltonian admits an *exact* MPO of small bond dimension whose local tensor is
block-upper-triangular in a virtual "memory" space,

$$ W_H = \begin{pmatrix} \mathbb{I} & C & D \\ 0 & A & B \\ 0 & 0 & \mathbb{I} \end{pmatrix}, $$

where $D$ holds the on-site terms, $C$/$B$ start/end an interaction, and $A$ propagates it
across intermediate sites. Reading left to right, the virtual index behaves like the internal state
of a finite-state machine that records whether an interaction has been started, carried, or
terminated. For the ANNNI-type model, the local tensor has bond dimension $D_W = 5$:

$$ W_H = \begin{pmatrix}
  \mathbb{I} & -\sigma^z & -p\lambda\,\sigma^x & -p\,\sigma^z & -\lambda\,\sigma^x \\
  0 & 0 & 0 & 0 & \sigma^z \\
  0 & 0 & 0 & 0 & \sigma^x \\
  0 & \mathbb{I} & 0 & 0 & 0 \\
  0 & 0 & 0 & 0 & \mathbb{I} \end{pmatrix}. $$

Note how this construction treats the NNN term: initiating it puts the machine into a dedicated
memory state (row 4) that applies an identity at the intermediate site $i+1$ and carries the
pending $\sigma^z$ untouched until the interaction terminates at $i+2$. Because the matrix is
strictly upper-triangular, the machine only ever reads forward. Therefore, the MPO is inherently
asymmetric (it breaks left–right parity). After the $90^\circ$ transverse rotation, this makes the
spatial transfer matrix non-Hermitian, which dictates the left–right power method we use later
(notebook 5). In practice, we never hand-build this matrix: `expmpo` derives the FSM automatically
from the `OpSum`.

## Exponentiating the FSM: WI, WII, VD2

Turning $W_H$ into $U(\delta t)$ is not trivial: $W_H$ is upper-triangular, so its powers mix the
blocks (e.g. $(\tau W_H)^2$ generates $CD, BC, AD,\dots$ with $\tau=-i\,\delta t$), and each
cross-term must appear with exactly the coefficient prescribed by the Taylor series of
$e^{\tau W_H}$. A further obstacle is size-extensivity: naively truncating
$e^{\tau H}=\sum_n \tau^n H^n/n!$ fails because $\|H^n|\Psi\rangle\|$ scales as $N^n$, so the sum
cannot be normalized in the thermodynamic limit. The fix is to organize the expansion by how the
local terms overlap along the chain (Zaletel et al. 2015; Van Damme et al. 2024).

Three kernels are available (selected by `mpo_alg`):

| kernel | order | virtual bond | notes |
|---|---|---|---|
| `WI`  | 1st | $1+\chi$ | $W_I \approx \mathbb{I}+\tau W_H$; keeps disconnected terms only. |
| `WII` | 1st (2nd for strictly-NN $H$) | $1+\chi$ | Zaletel two-boson construction; captures single-site overlaps. |
| `VD2` | 2nd (unconditionally) | $1+\chi+\chi^2$ | Van Damme Appendix-A. The one we use. |

The subtlety that matters for our NNN model: `WII` is only genuinely first order for a generic
Hamiltonian, becoming effectively second order only when every pair of local terms overlaps on
at most one site (strictly nearest-neighbour). Our local terms (the $\sigma^z\sigma^z_{i+2}$
bridge in particular) overlap on two sites, so `WII` is genuinely first order here, while `VD2`
is unconditionally second order. The price is a larger virtual bond ($1+\chi+\chi^2$ vs $1+\chi$),
which after the transverse rotation becomes a larger temporal physical dimension. We therefore
always read that dimension dynamically (never hard-code it).

### The VD2 block tensor (Van Damme et al. 2024, Appendix A)

With scaled blocks $\tilde D=\tau D$, $\tilde C=\sqrt{\tau}\,C$, $\tilde B=\sqrt{\tau}\,B$,
$\tilde A=A$, and $\{\cdots\}$ the symmetric sum over distinct permutations, the second-order
MPO is the $3\times3$ block tensor

$$\begin{aligned}
W_{11}&=\mathbb{I}+\tilde D+\tfrac12\{\tilde D\tilde D\}+\tfrac16\{\tilde D\tilde D\tilde D\}, &
W_{12}&=\tilde C+\tfrac12\{\tilde C\tilde D\}+\tfrac16\{\tilde C\tilde D\tilde D\}, &
W_{13}&=\{\tilde C\tilde C\}+\tfrac13\{\tilde C\tilde C\tilde D\},\\
W_{21}&=\tilde B+\tfrac12\{\tilde B\tilde D\}+\tfrac16\{\tilde B\tilde D\tilde D\}, &
W_{22}&=\tilde A+\tfrac12(\{\tilde B\tilde C\}+\{\tilde A\tilde D\})+\tfrac16(\{\tilde C\tilde B\tilde D\}+\{\tilde A\tilde D\tilde D\}), &
W_{23}&=\{\tilde A\tilde C\}+\tfrac13(\{\tilde A\tilde C\tilde D\}+\{\tilde C\tilde C\tilde B\}),\\
W_{31}&=\tfrac12\{\tilde B\tilde B\}+\tfrac16\{\tilde B\tilde B\tilde D\}, &
W_{32}&=\tfrac12\{\tilde A\tilde B\}+\tfrac16(\{\tilde A\tilde B\tilde D\}+\{\tilde B\tilde B\tilde C\}), &
W_{33}&=\{\tilde A\tilde A\}+\tfrac13(\{\tilde A\tilde B\tilde C\}+\{\tilde A\tilde A\tilde D\}).
\end{aligned}$$

This is exactly what the VD2 kernel (`makeW(::Algorithm"VD2", …)` in
`ITensorExpMPOv2.jl/src/eulerbuilder.jl`) transcribes. The single-vs-doubled memory leg (block
index 2 vs 3) is what enlarges the bond to $1+\chi+\chi^2$ and buys the clean order separation.

In [ ]:
include("../src/thesislib.jl")
using Printf 

N = 12; sites = siteinds("S=1/2", N); lambda = 1.0; p = 0.1; dt = 0.1
for alg in ("WI","WII","VD2")
    U = expH_alcaraz(sites, lambda, p; dt=dt, mpo_alg=alg)
    println(rpad(alg,4), "  max virtual bond dim = ", maxlinkdim(U))
end

WI    max virtual bond dim = 4
WII   max virtual bond dim = 4
VD2   max virtual bond dim = 13


## Schrödinger-picture benchmark against TDVP

Before using the MPO inside the transverse machinery, let us check it in the ordinary Schrödinger
picture: evolve $|X+\rangle^{\otimes N}$ by repeatedly applying $U(\delta t)$ and compare the
Loschmidt rate $\ell(T)=-\log|\mathcal{L}|/N$ against TDVP (which does not Trotterize and is our
ground truth).

In [ ]:
# Schrödinger benchmark (Alcaraz, N=40, dt=0.05): generate-or-load each rate curve, then plot.
# Each curve caches to results/data/rate_*.jld2 and is skipped if present (crash-safe / self-contained).
#   rate ℓ(T) = -log|⟨ψ₀|ψ(T)⟩| / N,   ψ(T) from many U(δt) applies (MPO kernel) vs from TDVP.
target_times = collect(0.5:1.0:6.5)
Nb = 40; dtb = 0.05
sb = siteinds("S=1/2", Nb); psi0b = complex(MPS(sb, "X+"))

# (a) TDVP ground truth — reuses the crash-safe TDVP driver (shares the N=40 Loschmidt cache)
function rate_tdvp()
    f = "../results/data/rate_TDVP.jld2"
    isfile(f) && return load(f, "rate_TDVP")
    done = tdvp_loschmidt_amplitude(Nb, target_times; p=0.1, lambda=1.0, dt=dtb,
                                    cachefile="../results/data/tdvp_loschmidt_p0.1_N40.jld2")
    r = [done[T].rate for T in target_times]
    jldopen(f, "w") do file; file["rate_TDVP"] = r; end
    r
end

# (b) direct exp-MPO apply rate for a kernel ("WII" or "VD2")
function rate_mpo(alg)
    f = "../results/data/rate_$(alg).jld2"; key = "rate_$(alg)"
    isfile(f) && return load(f, key)
    U = expH_alcaraz(sb, 1.0, 0.1; dt=dtb, mpo_alg=alg)
    r = Float64[]
    for T in target_times
        psi = deepcopy(psi0b)
        for _ in 1:round(Int, T/dtb); psi = apply(U, psi; cutoff=1e-14, maxdim=256); normalize!(psi); end
        push!(r, -log(abs(inner(psi0b, psi))) / Nb)
    end
    jldopen(f, "w") do file; file[key] = r; end
    r
end

r_TDVP = rate_tdvp(); r_VD2 = rate_mpo("VD2"); r_WII = rate_mpo("WII")

plt = plot(title="Schrödinger benchmark: exp-MPO vs TDVP", xlabel="T", ylabel="rate ℓ(T)",
           framestyle=:box, grid=true, legend=:topright)
plot!(plt, target_times, r_TDVP, lw=3, color=:black, label="TDVP (ground truth)")
plot!(plt, target_times, r_VD2,  lw=2, ls=:dash, marker=:circle, color=:blue, label="VD2 MPO")
plot!(plt, target_times, r_WII,  lw=2, ls=:dot,  marker=:cross,  color=:red,  label="WII MPO")
plt

- **VD2 is indistinguishable from TDVP** at every time, which validates both the MPO construction and the second-order
- **WII works at short times but deviates at larger $T$** (genuinely-first-order error of `WII` on a NNN Hamiltonian)

There is also a **fine balance** at play whenever we go to the transverse picture: shrinking
$\delta t$ lowers the Trotter error but lengthens the temporal chain, which raises the truncation
error of the long temporal MPS. One has to find the sweet spot for each quantity (see NB on entropies for more details)

## How unitary is the MPO? (the Trotter leak)

$U(\delta t)$ is only approximately unitary, so applying several layers to a normalized state
should preserve its norm up to Trotter error. A growing drift is the "Trotter leak" that
accumulates over long simulations. We check WI vs WII vs VD2 on a long chain.

In [2]:
# Local helper: apply `layers` copies of U(dt) to |X+>^N and report the norm drift.
function test_unitarity(N, lambda, p, dt, layers; mpo_alg="VD2", cutoff=1e-12, maxdim=256)
    sites = siteinds("S=1/2", N)
    U     = expH_alcaraz(sites, lambda, p; dt=dt, mpo_alg=mpo_alg)
    psi   = complex(MPS(sites, "X+"))
    println("--- $mpo_alg ---")
    for L in 1:layers
        psi = apply(U, psi; cutoff=cutoff, maxdim=maxdim)
        @printf("  layer %d   ‖ψ‖ = %.8f\n", L, norm(psi))
    end
end

N = 100; lambda = 1.0; p = 0.1; dt = 0.05; layers = 8
test_unitarity(N, lambda, p, dt, layers; mpo_alg="WI")
test_unitarity(N, lambda, p, dt, layers; mpo_alg="WII")
test_unitarity(N, lambda, p, dt, layers; mpo_alg="VD2")

--- WI ---
  layer 1   ‖ψ‖ = 1.35143355
  layer 2   ‖ψ‖ = 1.83028996
  layer 3   ‖ψ‖ = 2.48597368
  layer 4   ‖ψ‖ = 3.38993695
  layer 5   ‖ψ‖ = 4.64780885
  layer 6   ‖ψ‖ = 6.41895837
  layer 7   ‖ψ‖ = 8.94791080
  layer 8   ‖ψ‖ = 12.61458367
--- WII ---
  layer 1   ‖ψ‖ = 1.13410188
  layer 2   ‖ψ‖ = 1.28823723
  layer 3   ‖ψ‖ = 1.46762864
  layer 4   ‖ψ‖ = 1.67875251
  layer 5   ‖ψ‖ = 1.92943093
  layer 6   ‖ψ‖ = 2.22889275
  layer 7   ‖ψ‖ = 2.58778781
  layer 8   ‖ψ‖ = 3.01813815
--- VD2 ---
  layer 1   ‖ψ‖ = 0.99994500
  layer 2   ‖ψ‖ = 0.99988524
  layer 3   ‖ψ‖ = 0.99981626
  layer 4   ‖ψ‖ = 0.99973416
  layer 5   ‖ψ‖ = 0.99963580
  layer 6   ‖ψ‖ = 0.99951903
  layer 7   ‖ψ‖ = 0.99938271
  layer 8   ‖ψ‖ = 0.99922674


We observe that the Trotter leak for the VD2 MPO construction is considerably lower than in the other cases. Therefore, this construction represents what we were looking in the previous notebook: a single, uniform MPO column $U(\delta t)$, identical at every site, that reproduces the exact dynamics (VD2 $\equiv$ TDVP) (ingredient required for transverse contraction).

Stacking $U(\delta t)$ layers builds the 2D space-time network for
$\langle\Psi_0|e^{-iHT}|\Psi_0\rangle$. Rotating it by $90^\circ$ turns the virtual spatial
bonds into physical temporal sites and the physical spatial legs into the virtual
temporal bonds. A bulk column becomes the spatial transfer matrix $\mathcal{E}$, and (due to translation invariance) contracting along space reduces to finding the dominant left/right
eigenvectors of one $\mathcal{E}$ (the power method, see notebook 5).

It is important to note that (i) VD2's larger bond ($1+\chi+\chi^2$)
sets the temporal physical dimension, and (ii) the upper-triangular FSM makes $\mathcal{E}$
non-Hermitian/asymmetric, so its left and right fixed points differ.

Finally, a spoiler: even with the validated VD2 MPO, the transverse contraction starts to
break down beyond a certain time: the power method struggles to converge as a spectral gap
closes. Diagnosing that is the subject of notebook 5; first, notebooks 3 and 4 put the working
machinery to use on the temporal entropies and the equilibrium central charge.